# uq_setup
UQ experiment setup and execution notebook.

* gridkit installed in: `/home/isatkaus/gridkit`

#### workflow
1. run imports cell
2. run runner setup cell
3. run ONE case config cell (hawaii or illinois), then proceed with shared cells
4. create meta.yml, generate samples, create run dirs, run all, collect

#### references
* `/home/isatkaus/gridkit/uq-usecase/cases/hawaii.md` — hawaii UQ docs
* `/home/isatkaus/gridkit/uq-usecase/cases/illinois.md` — illinois UQ docs
* `/home/isatkaus/gridkit/uq-usecase/py-utils/gridkit_utils.py` — sampling + run utilities

In [1]:
import pandas as pd
import numpy as np
import os
import sys
import glob
import shutil
import datetime
import subprocess
import yaml

import plotly.express as px
from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"

pd.set_option("display.max_rows", 10)
pd.set_option("display.max_columns", 100)
pd.options.plotting.backend = "plotly"

# === GridKit paths ===
GRIDKIT_REPO_ROOT = os.path.expanduser("~/gridkit")
GRIDKIT_BUILD_DIR = os.path.join(GRIDKIT_REPO_ROOT, "build")
GRIDKIT_PY_UTILS = os.path.join(GRIDKIT_REPO_ROOT, "uq-usecase/py-utils")
if GRIDKIT_PY_UTILS not in sys.path:
    sys.path.insert(0, GRIDKIT_PY_UTILS)

print(f"GRIDKIT_REPO_ROOT: {GRIDKIT_REPO_ROOT}")
print(f"GRIDKIT_BUILD_DIR: {GRIDKIT_BUILD_DIR}")
print(f"GRIDKIT_PY_UTILS:  {GRIDKIT_PY_UTILS}")

GRIDKIT_REPO_ROOT: /home/isatkaus/gridkit
GRIDKIT_BUILD_DIR: /home/isatkaus/gridkit/build
GRIDKIT_PY_UTILS:  /home/isatkaus/gridkit/uq-usecase/py-utils


## runner setup

In [3]:
build_dir = GRIDKIT_BUILD_DIR
runner = os.path.join(build_dir, "application/PhasorDynamics/DynamicSimulation")
os.environ["PATH"] = os.path.dirname(runner) + os.pathsep + os.environ["PATH"]
print(f"runner: {runner}")
print(f"exists: {os.path.exists(runner)}")

runner: /home/isatkaus/gridkit/build/application/PhasorDynamics/DynamicSimulation
exists: True


In [4]:
!which DynamicSimulation

~/gridkit/build/application/PhasorDynamics/DynamicSimulation


## case config
Run ONE of the setup cells below:

> hawaii setup

> illinoi setup

then proceed with the shared cells (meta, samples, run, collect).

### monitored variables

`MONITORS_BY_CLASS` in each config cell controls which signals are recorded per device class.
`make_run_dir` **overwrites** the `mon` array in every matching bus/device entry of the copied case JSON with these values — the base case values (`Vr`/`Vi` by default) are replaced.

| Class | Variable | Description |
|-------|----------|-------------|
| `bus` | `Vm` | voltage magnitude (pu) |
| `bus` | `Va` | voltage angle (rad) |
| `bus` | `Vr` | voltage real part (pu) — base case default |
| `bus` | `Vi` | voltage imaginary part (pu) — base case default |
| `genrou` | `delta` | rotor angle (rad) |
| `genrou` | `omega` | speed deviation from synchronous (pu); steady-state ~0 |

Current choice: **polar** (`Vm`, `Va`) instead of the base case rectangular (`Vr`, `Vi`).


### hawaii setup

In [ ]:
# === UQ config (Hawaii) ===
import importlib
import gridkit_utils

importlib.reload(gridkit_utils)
from gridkit_utils import generate_samples, make_run_dir, run_sample, collect_and_save

# --- case files ---
BASE_CASE_DIR = os.path.join(build_dir, "examples/PhasorDynamics/Medium/Hawaii/")
CASE_JSON = os.path.join(BASE_CASE_DIR, "hawaii.json")
SOLVER_JSON = os.path.join(BASE_CASE_DIR, "hawaii.solver.json")

# --- output ---
UQ_RUN_ROOT = "/kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-v5"
# "per_run" (default): one run_NNN.parquet per run under runs/ -- use for N>=100
# "stacked": all runs in one results.parquet -- only for small N (<500)
SERIALIZE_MODE = "per_run"
UQ_OUT_PATH = (
    os.path.join(UQ_RUN_ROOT, "results.parquet")
    if SERIALIZE_MODE == "stacked"
    else os.path.join(UQ_RUN_ROOT, "runs")
)

# --- sampling ---
N_SAMPLES = 16000
SEED = 42
SAMPLE_METHOD = "lhs"  # "lhs" or "random"

# --- solver overrides ---
# None = use base case values unchanged
# tmax: simulation end time (s)
# events: list of {index, ...fields} -- patch events by 0-based index
SOLVER_OVERRIDES = None
# SOLVER_OVERRIDES = {
#     "tmax": 10.0,
#     "events": [
#         {"index": 0, "time": 1.0},   # fault_on  (base: 1.0s)
#         {"index": 1, "time": 1.1},   # fault_off (base: 1.1s)
#     ],
# }

# -----------------------------------------------------------------------
# H parameter specs: uncomment ONE block
# bus 2: H=3.69, bus 23: H=6.15, bus 34: H=4.35, bus 35: H=5.22
# -----------------------------------------------------------------------

# --- option A: aleatoric -- uniform +/-10% of nominal ---
# _H_PCT = 0.10
# PARAM_SPECS = [
#     {"class": "Genrou", "id": "2_1",  "param": "H", "dist": "uniform", "nominal": 3.69, "pct": _H_PCT},
#     {"class": "Genrou", "id": "23_1", "param": "H", "dist": "uniform", "nominal": 6.15, "pct": _H_PCT},
#     {"class": "Genrou", "id": "34_1", "param": "H", "dist": "uniform", "nominal": 4.35, "pct": _H_PCT},
#     {"class": "Genrou", "id": "35_1", "param": "H", "dist": "uniform", "nominal": 5.22, "pct": _H_PCT},
# ]

# --- option B: epistemic -- Gaussian std=12% of nominal ---
_H_STD_PCT = 0.12
PARAM_SPECS = [
    {
        "class": "Genrou",
        "id": "2_1",
        "param": "H",
        "dist": "normal",
        "mean": 3.69,
        "std": 3.69 * _H_STD_PCT,
    },
    {
        "class": "Genrou",
        "id": "23_1",
        "param": "H",
        "dist": "normal",
        "mean": 6.15,
        "std": 6.15 * _H_STD_PCT,
    },
    {
        "class": "Genrou",
        "id": "34_1",
        "param": "H",
        "dist": "normal",
        "mean": 4.35,
        "std": 4.35 * _H_STD_PCT,
    },
    {
        "class": "Genrou",
        "id": "35_1",
        "param": "H",
        "dist": "normal",
        "mean": 5.22,
        "std": 5.22 * _H_STD_PCT,
    },
]

# -----------------------------------------------------------------------
# Monitored variables: make_run_dir patches the mon[] array in each
# bus/device entry of the copied case JSON with these values, overwriting
# the base case defaults (Vr, Vi).
#
# bus:    Vm = voltage magnitude (pu), Va = voltage angle (rad)
#         alternatives: Vr = real part, Vi = imaginary part
# genrou: delta = rotor angle (rad), omega = speed deviation (pu, ~0 steady-state)
# -----------------------------------------------------------------------
MONITORS_BY_CLASS = {
    "bus": ["Vm", "Va"],
    "genrou": ["delta", "omega"],
}

print(f"BASE_CASE_DIR: {BASE_CASE_DIR}")
print(f"CASE_JSON:     {CASE_JSON}  (exists: {os.path.exists(CASE_JSON)})")
print(f"SOLVER_JSON:   {SOLVER_JSON}  (exists: {os.path.exists(SOLVER_JSON)})")
print(f"UQ_RUN_ROOT:   {UQ_RUN_ROOT}")
print(f"SERIALIZE_MODE:{SERIALIZE_MODE}")
print(f"N_SAMPLES:     {N_SAMPLES}")
_dist = PARAM_SPECS[0]["dist"]
if _dist == "uniform":
    print(f"H dist: uniform +/-{PARAM_SPECS[0]['pct']*100:.0f}% of nominal")
    for s in PARAM_SPECS:
        print(
            f"  {s['id']}: nominal={s['nominal']:.2f}  [{s['nominal']*(1-s['pct']):.4f}, {s['nominal']*(1+s['pct']):.4f}]"
        )
else:
    print(
        f"H dist: normal, std={PARAM_SPECS[0]['std']/PARAM_SPECS[0]['mean']*100:.0f}% of nominal"
    )
    for s in PARAM_SPECS:
        print(f"  {s['id']}: mean={s['mean']:.2f}, std={s['std']:.4f}")

print(f"\n--- {os.path.basename(SOLVER_JSON)} ---")
print(open(SOLVER_JSON).read())

<module 'gridkit_utils' from '/home/isatkaus/gridkit/uq-usecase/py-utils/gridkit_utils.py'>

BASE_CASE_DIR: /home/isatkaus/gridkit/build/examples/PhasorDynamics/Medium/Hawaii/
CASE_JSON:     /home/isatkaus/gridkit/build/examples/PhasorDynamics/Medium/Hawaii/hawaii.json  (exists: True)
SOLVER_JSON:   /home/isatkaus/gridkit/build/examples/PhasorDynamics/Medium/Hawaii/hawaii.solver.json  (exists: True)
UQ_RUN_ROOT:   /kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-v5
SERIALIZE_MODE:per_run
N_SAMPLES:     16000
H dist: normal, std=12% of nominal
  2_1: mean=3.69, std=0.4428
  23_1: mean=6.15, std=0.7380
  34_1: mean=4.35, std=0.5220
  35_1: mean=5.22, std=0.6264

--- hawaii.solver.json ---
{
    "system_model_file": "hawaii.json",
    "dt": 0.00416666666666,
    "tmax": 10.0,
    "events": [
        { "time": 1.0, "type": "fault_on", "element_id": 0 },
        { "time": 1.1, "type": "fault_off", "element_id": 0 }
    ]
}



### illinois setup

In [ ]:
# === UQ config (Illinois) ===
import importlib
import gridkit_utils

importlib.reload(gridkit_utils)
from gridkit_utils import generate_samples, make_run_dir, run_sample, collect_and_save

# --- case files ---
BASE_CASE_DIR = os.path.join(build_dir, "examples/PhasorDynamics/Large/Illinois/")
CASE_JSON = os.path.join(BASE_CASE_DIR, "illinois.json")
SOLVER_JSON = os.path.join(BASE_CASE_DIR, "illinois.solver.json")

# --- output ---
UQ_RUN_ROOT = "/kfs2/projects/scidac/scidac-data/gridkit-runs/illinois-v2"
# "per_run" (default): one run_NNN.parquet per run under runs/ -- use for N>=100
# "stacked": all runs in one results.parquet -- only for small N (<500)
SERIALIZE_MODE = "per_run"
UQ_OUT_PATH = (
    os.path.join(UQ_RUN_ROOT, "results.parquet")
    if SERIALIZE_MODE == "stacked"
    else os.path.join(UQ_RUN_ROOT, "runs")
)

# --- sampling ---
N_SAMPLES = 4000
SEED = 42
SAMPLE_METHOD = "lhs"  # "lhs" or "random"

# --- solver overrides ---
# base illinois.solver.json: tmax=20.0 s, fault at t=10.0/10.1 s
# override to match Hawaii cadence: fault early, shorter sim
SOLVER_OVERRIDES = {
    "tmax": 10.0,
    "events": [
        {"index": 0, "time": 1.0},  # fault_on
        {"index": 1, "time": 1.1},  # fault_off
    ],
}

# -----------------------------------------------------------------------
# H parameter specs: 4 generators at increasing hop distance from fault (bus 2)
# hop 4: 126_1 BARTONVILLE 3  (coal, 130 MW)  H=7.29971
# hop 5: 135_1 PEKIN 1 2      (coal, 446 MW)  H=3.23290
# hop 7: 115_1 NORMAL 2 3     (wind, 133 MW)  H=2.72678
# hop 9: 189_1 CLINTON 1 2    (nuclear, 569 MW) H=3.40165
# -----------------------------------------------------------------------

# --- epistemic -- Gaussian std=12% of nominal ---
_H_STD_PCT = 0.12
PARAM_SPECS = [
    {
        "class": "Genrou",
        "id": "126_1",
        "param": "H",
        "dist": "normal",
        "mean": 7.29971075,
        "std": 7.29971075 * _H_STD_PCT,
    },
    {
        "class": "Genrou",
        "id": "135_1",
        "param": "H",
        "dist": "normal",
        "mean": 3.23289871,
        "std": 3.23289871 * _H_STD_PCT,
    },
    {
        "class": "Genrou",
        "id": "115_1",
        "param": "H",
        "dist": "normal",
        "mean": 2.72677827,
        "std": 2.72677827 * _H_STD_PCT,
    },
    {
        "class": "Genrou",
        "id": "189_1",
        "param": "H",
        "dist": "normal",
        "mean": 3.40165448,
        "std": 3.40165448 * _H_STD_PCT,
    },
]

# -----------------------------------------------------------------------
# Monitored variables: make_run_dir patches the mon[] array in each
# bus/device entry of the copied case JSON with these values, overwriting
# the base case defaults (Vr, Vi).
#
# bus:    Vm = voltage magnitude (pu), Va = voltage angle (rad)
#         alternatives: Vr = real part, Vi = imaginary part
# genrou: delta = rotor angle (rad), omega = speed deviation (pu, ~0 steady-state)
# -----------------------------------------------------------------------
MONITORS_BY_CLASS = {
    "bus": ["Vm", "Va"],
    "genrou": ["delta", "omega"],
}

print(f"BASE_CASE_DIR: {BASE_CASE_DIR}")
print(f"CASE_JSON:     {CASE_JSON}  (exists: {os.path.exists(CASE_JSON)})")
print(f"SOLVER_JSON:   {SOLVER_JSON}  (exists: {os.path.exists(SOLVER_JSON)})")
print(f"UQ_RUN_ROOT:   {UQ_RUN_ROOT}")
print(f"SERIALIZE_MODE:{SERIALIZE_MODE}")
print(f"N_SAMPLES:     {N_SAMPLES}")
print(f"SOLVER_OVERRIDES: {SOLVER_OVERRIDES}")
print(f"H dist: normal, std={_H_STD_PCT*100:.0f}% of nominal")
for s in PARAM_SPECS:
    print(f"  {s['id']}: mean={s['mean']:.5f}, std={s['std']:.5f}")

print(f"\n--- {os.path.basename(SOLVER_JSON)} (base, before overrides) ---")
print(open(SOLVER_JSON).read())

<module 'gridkit_utils' from '/home/isatkaus/gridkit/uq-usecase/py-utils/gridkit_utils.py'>

BASE_CASE_DIR: /home/isatkaus/gridkit/build/examples/PhasorDynamics/Large/Illinois/
CASE_JSON:     /home/isatkaus/gridkit/build/examples/PhasorDynamics/Large/Illinois/illinois.json  (exists: True)
SOLVER_JSON:   /home/isatkaus/gridkit/build/examples/PhasorDynamics/Large/Illinois/illinois.solver.json  (exists: True)
UQ_RUN_ROOT:   /kfs2/projects/scidac/scidac-data/gridkit-runs/illinois-v2
SERIALIZE_MODE:per_run
N_SAMPLES:     4000
SOLVER_OVERRIDES: {'tmax': 10.0, 'events': [{'index': 0, 'time': 1.0}, {'index': 1, 'time': 1.1}]}
H dist: normal, std=12% of nominal
  126_1: mean=7.29971, std=0.87597
  135_1: mean=3.23290, std=0.38795
  115_1: mean=2.72678, std=0.32721
  189_1: mean=3.40165, std=0.40820

--- illinois.solver.json (base, before overrides) ---
{
    "system_model_file": "illinois.json",
    "dt": 0.00416666666666,
    "tmax": 20.0,
    "events": [
        { "time": 10.0, "type": "fault_on", "element_id": 0 },
        { "time": 10.1, "type": "fault_off", "element_id": 0 }
    ]

## experiment steps

Run these cells in order after selecting a case config above.

1. Write `meta.yml`
2. Generate `samples.csv`
3. Create run dirs (patch `case.json` + `solver.json` for each sample)
4. **Run simulations** — choose ONE option:
   - **Serial** (next section): run all samples in a loop on this node; fine for small N or quick tests
   - **SLURM** (section after): write + submit bash array jobs; use for large N or large cases
5. Once all simulations complete, continue with **collect results** (shared regardless of run method)

In [11]:
# === Write experiment metadata to meta.yml ===
os.makedirs(UQ_RUN_ROOT, exist_ok=True)

_dist = PARAM_SPECS[0]["dist"]
_sampling_summary = []
for s in PARAM_SPECS:
    if s["dist"] == "uniform":
        lo = s["nominal"] * (1 - s["pct"])
        hi = s["nominal"] * (1 + s["pct"])
        _sampling_summary.append(
            {
                "id": s["id"],
                "dist": "uniform",
                "nominal": s["nominal"],
                "pct": s["pct"],
                "lo": round(lo, 6),
                "hi": round(hi, 6),
            }
        )
    else:
        _sampling_summary.append(
            {
                "id": s["id"],
                "dist": "normal",
                "mean": s["mean"],
                "std": round(s["std"], 6),
                "std_pct": round(s["std"] / s["mean"], 4),
            }
        )

meta = {
    "case": os.path.basename(BASE_CASE_DIR.rstrip("/")),
    "base_case_dir": BASE_CASE_DIR,
    "run_root": UQ_RUN_ROOT,
    "created": datetime.datetime.now().isoformat(timespec="seconds"),
    "sampling": {
        "n_samples": N_SAMPLES,
        "seed": SEED,
        "method": SAMPLE_METHOD,
        "dist_type": _dist,
        "params": _sampling_summary,
    },
    "serialize_mode": SERIALIZE_MODE,
    "solver_overrides": SOLVER_OVERRIDES,
    "monitors_by_class": MONITORS_BY_CLASS,
}
meta_path = os.path.join(UQ_RUN_ROOT, "meta.yml")
with open(meta_path, "w") as f:
    yaml.dump(meta, f, default_flow_style=False, sort_keys=False)
print(f"Wrote {meta_path}")
print(open(meta_path).read())

Wrote /kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-v5/meta.yml
case: Hawaii
base_case_dir: /home/isatkaus/gridkit/build/examples/PhasorDynamics/Medium/Hawaii/
run_root: /kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-v5
created: '2026-07-22T10:39:40'
sampling:
  n_samples: 16000
  seed: 42
  method: lhs
  dist_type: normal
  params:
  - id: '2_1'
    dist: normal
    mean: 3.69
    std: 0.4428
    std_pct: 0.12
  - id: '23_1'
    dist: normal
    mean: 6.15
    std: 0.738
    std_pct: 0.12
  - id: '34_1'
    dist: normal
    mean: 4.35
    std: 0.522
    std_pct: 0.12
  - id: '35_1'
    dist: normal
    mean: 5.22
    std: 0.6264
    std_pct: 0.12
serialize_mode: per_run
solver_overrides: null
monitors_by_class:
  bus:
  - Vm
  - Va
  genrou:
  - delta
  - omega



In [12]:
# === Generate samples ===
os.makedirs(UQ_RUN_ROOT, exist_ok=True)
samples_df = generate_samples(PARAM_SPECS, N=N_SAMPLES, seed=SEED, method=SAMPLE_METHOD)
samples_df.to_csv(os.path.join(UQ_RUN_ROOT, "samples.csv"))
print(f"samples_df shape: {samples_df.shape}")
samples_df

samples_df shape: (16000, 4)


,Genrou_2_1_H,Genrou_23_1_H,Genrou_34_1_H,Genrou_35_1_H
0,3.982270,6.904589,3.883889,5.402560
1,3.742142,6.023595,4.705813,6.008303
2,3.786765,5.666869,4.060240,5.501311
3,3.478043,4.848891,4.042315,5.523162
4,3.786309,5.085319,3.439594,5.427600
...,...,...,...,...
15995,3.896439,5.211398,4.720589,5.161782
15996,3.288781,5.586668,3.913647,4.380928
15997,3.998747,5.587183,4.440567,5.521512
15998,3.415513,6.410713,4.335614,4.000952


In [13]:
# === Create run dirs + patch case.json ===
for i, row in samples_df.iterrows():
    d = make_run_dir(
        BASE_CASE_DIR,
        UQ_RUN_ROOT,
        i,
        row,
        PARAM_SPECS,
        MONITORS_BY_CLASS,
        solver_overrides=SOLVER_OVERRIDES,
    )
    if i % 100 == 0:
        print(f"  run_{i:03d}: {d}")
print(f"Created {N_SAMPLES} run dirs in {UQ_RUN_ROOT}")

  run_000: /kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-v5/run_000
  run_100: /kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-v5/run_100
  run_200: /kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-v5/run_200
  run_300: /kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-v5/run_300
  run_400: /kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-v5/run_400
  run_500: /kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-v5/run_500
  run_600: /kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-v5/run_600
  run_700: /kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-v5/run_700
  run_800: /kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-v5/run_800
  run_900: /kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-v5/run_900
  run_1000: /kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-v5/run_1000
  run_1100: /kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-v5/run_1100
  run_1200: /kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-v5/run_1200
  run_

### option A: serial run

Run all samples in a loop on this node.
Good for small N (up to ~200) or quick tests.
**Skip this cell** and go to option B below for large N or large grid cases.

In [28]:
# === Run all samples ===
failed = []
for i, row in samples_df.iterrows():
    run_dir = os.path.join(UQ_RUN_ROOT, f"run_{i:03d}")
    result = run_sample(run_dir, runner)
    status = "OK" if result.returncode == 0 else "FAILED"
    if result.returncode != 0:
        failed.append(i)
        print(f"  run_{i:03d}: FAILED  {result.stderr[:200]}")
    elif i % 100 == 0:
        print(f"  run_{i:03d}: OK")

print(f"\nDone. {N_SAMPLES - len(failed)}/{N_SAMPLES} succeeded.")
if failed:
    print(f"Failed runs: {failed}")

  run_000: OK
  run_100: OK
  run_200: OK
  run_300: OK
  run_400: OK
  run_500: OK
  run_600: OK
  run_700: OK
  run_800: OK
  run_900: OK

Done. 1000/1000 succeeded.


### option B: SLURM parallel run

Run samples as bash array jobs across multiple cluster nodes.
Use for large N (1K+) or large grid cases where serial is too slow.
**Skip this section** if you ran option A above.

Steps:
1. Set SLURM config (nodes, partition, account, walltime, workers per node)
2. Write sbatch scripts — one bash script per node, each loops over its sample slice using `xargs -P`
3. Submit all scripts with `sbatch`
4. Re-run the status cell until all jobs complete, then continue with **collect results** below

#### SLURM config

In [16]:
# === SLURM config ===
SLURM_ACCOUNT = "msoc"  # --account  (your project handle)
SLURM_PARTITION = "short"  # --partition
WALLTIME = "03:00:00"  # --time
N_NODES = 4  # number of sbatch scripts (one per node)
WORKERS_PER_NODE = 4  # parallel DynamicSimulation per node via xargs -P
# keep <=104 (Kestrel CPUs/node); use 1 for large grids
SLURM_QOS = None  # None = default priority
# "high"    = 2x AU cost, faster queue
# "standby" = free, only runs on idle nodes

# Modules to load inside each sbatch script.
# Add whatever DynamicSimulation needs at runtime (e.g. MKL, compiler libs).
# Leave empty if the binary is self-contained or the environment is set via conda.
SLURM_MODULES = [
    # "intel-oneapi-mkl/2024.0",
    # "gcc/13.1.0",
]

# --- collect job (option B: SLURM-side collection) ---
# If True, a separate collect.sh job is written and submitted with --dependency=afterok
# on all sim jobs. It runs on one additional node after all sims complete.
# If False, collect manually using the notebook collect cell below.
COLLECT_IN_SLURM = True
COLLECT_WORKERS = 64  # threads for SLURM collect; more aggressive than notebook
COLLECT_WALLTIME = "02:00:00"
# conda env path used by the collect job to import pandas + pyarrow
CONDA_ENV = "/home/isatkaus/flash-w-dom/conda-envs/h-py312-basic"

# Derived: slice indices per node
_chunk_size = (N_SAMPLES + N_NODES - 1) // N_NODES
_slices = [
    (i * _chunk_size, min((i + 1) * _chunk_size - 1, N_SAMPLES - 1))
    for i in range(N_NODES)
]
print(f"N_SAMPLES={N_SAMPLES}, N_NODES={N_NODES}, WORKERS_PER_NODE={WORKERS_PER_NODE}")
print(f"QOS: {SLURM_QOS or '(default)'}")
print(f"COLLECT_IN_SLURM: {COLLECT_IN_SLURM}")
print("Index slices per node:")
for node_i, (lo, hi) in enumerate(_slices):
    print(f"  node {node_i:02d}: runs {lo:04d} - {hi:04d}  ({hi - lo + 1} samples)")
if COLLECT_IN_SLURM:
    print(f"  + collect node: {COLLECT_WORKERS} threads, walltime={COLLECT_WALLTIME}")

N_SAMPLES=16000, N_NODES=4, WORKERS_PER_NODE=4
QOS: (default)
COLLECT_IN_SLURM: True
Index slices per node:
  node 00: runs 0000 - 3999  (4000 samples)
  node 01: runs 4000 - 7999  (4000 samples)
  node 02: runs 8000 - 11999  (4000 samples)
  node 03: runs 12000 - 15999  (4000 samples)
  + collect node: 64 threads, walltime=02:00:00


In [17]:
# === Write sbatch scripts ===
import stat

_slurm_dir = os.path.join(UQ_RUN_ROOT, "slurm")
os.makedirs(_slurm_dir, exist_ok=True)

_module_lines = (
    "\n".join(f"module load {m}" for m in SLURM_MODULES)
    if SLURM_MODULES
    else "# (no modules configured)"
)
_qos_line = (
    f"#SBATCH --qos={SLURM_QOS}"
    if SLURM_QOS
    else '#SBATCH --qos=normal  # (default; set SLURM_QOS="high" for priority)'
)

# Run dirs use 3-digit format: run_000, run_001, ...  (matches make_run_dir f"run_{i:03d}")
_script_paths = []
for node_i, (lo, hi) in enumerate(_slices):
    script_path = os.path.join(_slurm_dir, f"chunk_{node_i:02d}.sh")
    _indices = " ".join(f"{i:03d}" for i in range(lo, hi + 1))
    script = f"""#!/bin/bash
#SBATCH --job-name=uq_{os.path.basename(UQ_RUN_ROOT)}_n{node_i:02d}
#SBATCH --account={SLURM_ACCOUNT}
#SBATCH --partition={SLURM_PARTITION}
{_qos_line}
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --time={WALLTIME}
#SBATCH --output={_slurm_dir}/chunk_{node_i:02d}_%j.out
#SBATCH --error={_slurm_dir}/chunk_{node_i:02d}_%j.err

{_module_lines}

# do not use -e: individual run failures must not abort the whole chunk
set -uo pipefail
cd {UQ_RUN_ROOT}
FAILED={_slurm_dir}/chunk_{node_i:02d}_failed.txt
rm -f "$FAILED"

INDICES=({_indices})
printf '%s\\n' "${{INDICES[@]}}" | xargs -P {WORKERS_PER_NODE} -I {{}} bash -c '
    dir="run_{{}}"
    solver=$(ls "$dir"/*.solver.json 2>/dev/null | head -1)
    if [[ -z "$solver" ]]; then
        echo "NO_SOLVER $dir" >> "{_slurm_dir}/chunk_{node_i:02d}_failed.txt"
        exit 0
    fi
    cd "$dir"
    {runner} "$(basename $solver)" > stdout.txt 2> stderr.txt \\
        || echo "FAILED $dir" >> "{_slurm_dir}/chunk_{node_i:02d}_failed.txt"
'
echo "chunk {node_i:02d} done"
"""
    with open(script_path, "w") as fh:
        fh.write(script)
    os.chmod(script_path, os.stat(script_path).st_mode | stat.S_IXUSR | stat.S_IXGRP)
    _script_paths.append(script_path)
    print(f"  wrote {script_path}  (runs {lo:03d}-{hi:03d})")

print(f"\n{len(_script_paths)} sim sbatch scripts written to {_slurm_dir}/")

# --- optional collect job ---
_collect_script_path = None
if COLLECT_IN_SLURM:
    _collect_script_path = os.path.join(_slurm_dir, "collect.sh")
    _collect_script = f"""#!/bin/bash
#SBATCH --job-name=uq_{os.path.basename(UQ_RUN_ROOT)}_collect
#SBATCH --account={SLURM_ACCOUNT}
#SBATCH --partition={SLURM_PARTITION}
{_qos_line}
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --time={COLLECT_WALLTIME}
#SBATCH --output={_slurm_dir}/collect_%j.out
#SBATCH --error={_slurm_dir}/collect_%j.err

ml conda
conda activate {CONDA_ENV}

python3 - << 'PYEOF'
import sys
sys.path.insert(0, "{GRIDKIT_PY_UTILS}")
from gridkit_utils import collect_parallel

collect_parallel(
    run_root="{UQ_RUN_ROOT}",
    out_path="{UQ_OUT_PATH}",
    n_workers={COLLECT_WORKERS},
)
PYEOF
"""
    with open(_collect_script_path, "w") as fh:
        fh.write(_collect_script)
    os.chmod(
        _collect_script_path,
        os.stat(_collect_script_path).st_mode | stat.S_IXUSR | stat.S_IXGRP,
    )
    print(f"  wrote {_collect_script_path}  (collect job, {COLLECT_WORKERS} threads)")
    print(f"\nTotal nodes: {N_NODES} sim + 1 collect = {N_NODES + 1}")

print(f"\nReview sim script with:  cat {_script_paths[0]}")
if _collect_script_path:
    print(f"Review collect script:   cat {_collect_script_path}")

20304

  wrote /kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-v5/slurm/chunk_00.sh  (runs 000-3999)


21304

  wrote /kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-v5/slurm/chunk_01.sh  (runs 4000-7999)


23304

  wrote /kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-v5/slurm/chunk_02.sh  (runs 8000-11999)


25304

  wrote /kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-v5/slurm/chunk_03.sh  (runs 12000-15999)

4 sim sbatch scripts written to /kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-v5/slurm/


834

  wrote /kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-v5/slurm/collect.sh  (collect job, 64 threads)

Total nodes: 4 sim + 1 collect = 5

Review sim script with:  cat /kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-v5/slurm/chunk_00.sh
Review collect script:   cat /kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-v5/slurm/collect.sh


In [18]:
# === Submit sbatch jobs ===
# Review the scripts in slurm/ before running this cell.

_job_ids = []
for script_path in _script_paths:
    cmd = ["sbatch", script_path]
    print(" ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"  ERROR: {result.stderr.strip()}")
    else:
        jid = result.stdout.strip().split()[-1]
        _job_ids.append(jid)
        print(f"  submitted job {jid}")

print(f"\nSubmitted {len(_job_ids)} sim jobs: {_job_ids}")
print(
    f"Monitor with:  sacct -j {','.join(_job_ids)} --format=JobID,State,Elapsed,ExitCode"
)

# --- submit collect job with dependency if COLLECT_IN_SLURM ---
_collect_job_id = None
if COLLECT_IN_SLURM and _collect_script_path and _job_ids:
    _dep = "afterok:" + ":".join(_job_ids)
    cmd = ["sbatch", f"--dependency={_dep}", _collect_script_path]
    print(f"\n{' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"  ERROR submitting collect job: {result.stderr.strip()}")
    else:
        _collect_job_id = result.stdout.strip().split()[-1]
        print(
            f"  submitted collect job {_collect_job_id}  (runs after all sim jobs complete)"
        )
        print(f"\nAll jobs: sim={_job_ids}, collect={_collect_job_id}")
        print(f"Total: {len(_job_ids) + 1} nodes")

sbatch /kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-v5/slurm/chunk_00.sh
  submitted job 15345297
sbatch /kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-v5/slurm/chunk_01.sh
  submitted job 15345298
sbatch /kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-v5/slurm/chunk_02.sh
  submitted job 15345299
sbatch /kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-v5/slurm/chunk_03.sh
  submitted job 15345300

Submitted 4 sim jobs: ['15345297', '15345298', '15345299', '15345300']
Monitor with:  sacct -j 15345297,15345298,15345299,15345300 --format=JobID,State,Elapsed,ExitCode

sbatch --dependency=afterok:15345297:15345298:15345299:15345300 /kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-v5/slurm/collect.sh
  submitted collect job 15345301  (runs after all sim jobs complete)

All jobs: sim=['15345297', '15345298', '15345299', '15345300'], collect=15345301
Total: 5 nodes


In [20]:
# === Check job status ===
# Re-run this cell to poll until all jobs complete.
# Once all show COMPLETED with no failures, proceed to the collect cell.

# --- sacct status (requires jobs to have run) ---
_all_ids = list(_job_ids)
if COLLECT_IN_SLURM and "_collect_job_id" in dir() and _collect_job_id:
    _all_ids.append(_collect_job_id)

if _all_ids:
    r = subprocess.run(
        [
            "sacct",
            "-j",
            ",".join(_all_ids),
            "--format=JobID,JobName,State,Elapsed,ExitCode",
            "--noheader",
        ],
        capture_output=True,
        text=True,
    )
    print("sacct output:")
    print(r.stdout or "  (no output yet -- jobs may still be pending)")

# --- scan failed.txt files from each chunk ---
print("\nFailed runs (from chunk *_failed.txt files):")
_all_failed = []
for node_i in range(N_NODES):
    fpath = os.path.join(_slurm_dir, f"chunk_{node_i:02d}_failed.txt")
    if os.path.exists(fpath):
        lines = open(fpath).read().strip().splitlines()
        if lines:
            _all_failed.extend(lines)
            print(f"  chunk {node_i:02d}: {lines}")
        else:
            print(f"  chunk {node_i:02d}: OK (empty failed file)")
    else:
        print(f"  chunk {node_i:02d}: no failed.txt yet (pending or running)")

_still_running = "RUNNING" in r.stdout or "PENDING" in r.stdout
if _still_running:
    print(
        "\nJobs still in progress (RUNNING or PENDING). Re-run this cell to poll again."
    )
elif not _all_failed:
    print(
        "\nAll jobs completed. No failures detected. Safe to proceed with collect cell."
    )
else:
    print(
        f"\nAll jobs completed. {len(_all_failed)} failed run(s) -- investigate before collecting."
    )

sacct output:
15345297     uq_hawaii+  COMPLETED   00:05:13      0:0 
15345297.ba+      batch  COMPLETED   00:05:13      0:0 
15345297.ex+     extern  COMPLETED   00:05:13      0:0 
15345298     uq_hawaii+  COMPLETED   00:05:15      0:0 
15345298.ba+      batch  COMPLETED   00:05:15      0:0 
15345298.ex+     extern  COMPLETED   00:05:15      0:0 
15345299     uq_hawaii+  COMPLETED   00:05:10      0:0 
15345299.ba+      batch  COMPLETED   00:05:10      0:0 
15345299.ex+     extern  COMPLETED   00:05:10      0:0 
15345300     uq_hawaii+  COMPLETED   00:05:13      0:0 
15345300.ba+      batch  COMPLETED   00:05:13      0:0 
15345300.ex+     extern  COMPLETED   00:05:13      0:0 
15345301     uq_hawaii+  COMPLETED   00:06:16      0:0 
15345301.ba+      batch  COMPLETED   00:06:16      0:0 
15345301.ex+     extern  COMPLETED   00:06:16      0:0 


Failed runs (from chunk *_failed.txt files):
  chunk 00: no failed.txt yet (pending or running)
  chunk 01: no failed.txt yet (pending or runnin

## collect results

If `COLLECT_IN_SLURM=True`, the collect job was submitted automatically and will run after all sim jobs complete — skip the collect cell below and go straight to **results size** once the collect job shows COMPLETED in the status cell.

If `COLLECT_IN_SLURM=False` (default), run the collect cell below after all simulations complete.

**Note on threading:** `collect_parallel()` uses a `ThreadPoolExecutor`. The GIL is released during file I/O, so threads genuinely overlap on Lustre reads/writes. Bottleneck is Lustre MDS metadata throughput, not CPU. Safe range on Kestrel: 32 (conservative) to 64 (aggressive).


In [ ]:
# === Collect results -> Parquet ===
#
# SERIALIZE_MODE options (set in config cell above):
#   "stacked"  - all runs in one results.parquet; good for N<500, single-df analysis
#   "per_run"  - one run_NNN.parquet per run under runs/; good for large N (1000+)
#
# For per_run mode, uses collect_parallel() from gridkit_utils -- ThreadPoolExecutor,
# I/O-bound, GIL released during file reads/writes; bottleneck is Lustre MDS, not CPU.
# Kestrel Lustre safe range: 32 (conservative) to 64 (aggressive); above 64 rarely helps.
COLLECT_WORKERS = 32

if SERIALIZE_MODE == "stacked":
    result = collect_and_save(UQ_RUN_ROOT, samples_df, UQ_OUT_PATH, mode=SERIALIZE_MODE)
    results_df = result
    print(f"results_df: {results_df.shape}")
    results_df
else:
    from gridkit_utils import collect_parallel

    result, _missing = collect_parallel(
        UQ_RUN_ROOT, UQ_OUT_PATH, n_workers=COLLECT_WORKERS
    )
    for p in result[:5]:
        print(f"  {p}")
    if len(result) > 5:
        print(f"  ... ({len(result) - 5} more)")

Loaded samples_df from /kfs2/projects/scidac/scidac-data/gridkit-runs/illinois-v2/samples.csv  shape: (4000, 4)
  500/4000 processed, 500 written, 0 missing
  1000/4000 processed, 1000 written, 0 missing
  1500/4000 processed, 1500 written, 0 missing
  2000/4000 processed, 2000 written, 0 missing
  2500/4000 processed, 2500 written, 0 missing
  3000/4000 processed, 3000 written, 0 missing
  3500/4000 processed, 3500 written, 0 missing
  4000/4000 processed, 4000 written, 0 missing

Done. Written 4000 per-run files to /kfs2/projects/scidac/scidac-data/gridkit-runs/illinois-v2/runs
  /kfs2/projects/scidac/scidac-data/gridkit-runs/illinois-v2/runs/run_000.parquet
  /kfs2/projects/scidac/scidac-data/gridkit-runs/illinois-v2/runs/run_001.parquet
  /kfs2/projects/scidac/scidac-data/gridkit-runs/illinois-v2/runs/run_002.parquet
  /kfs2/projects/scidac/scidac-data/gridkit-runs/illinois-v2/runs/run_003.parquet
  /kfs2/projects/scidac/scidac-data/gridkit-runs/illinois-v2/runs/run_004.parquet
  .

## results size + data quality

Reports disk/memory footprint and spot-checks a sample of runs for missing files, NaNs, column consistency, and soft physical bounds.


In [ ]:
# === Results size: in-memory and on-disk ===
if SERIALIZE_MODE == "stacked":
    mem_mb = results_df.memory_usage(deep=True).sum() / 1024**2
    disk_mb = os.path.getsize(UQ_OUT_PATH) / 1024**2
    print(f"results_df:  {results_df.shape[0]:,} rows x {results_df.shape[1]} cols")
    print(f"  in-memory: {mem_mb:.1f} MB")
    print(f"  parquet on disk: {disk_mb:.2f} MB")
else:
    run_files = sorted(glob.glob(os.path.join(UQ_OUT_PATH, "run_*.parquet")))
    total_disk_mb = sum(os.path.getsize(f) for f in run_files) / 1024**2
    print(f"per_run: {len(run_files)} files in {UQ_OUT_PATH}")
    if run_files:
        print(
            f"  total on disk: {total_disk_mb:.2f} MB  ({total_disk_mb/len(run_files):.2f} MB/run)"
        )

# === Data quality check ===
# Spot-checks QC_SAMPLE_N randomly chosen runs for completeness, NaNs,
# column consistency, and soft physical bounds (warnings only, not failures).
QC_SAMPLE_N = 100

# Soft bounds -- wide enough to allow fault transients, just catches diverged/corrupt runs.
# Note: omega is speed *deviation* (delta-omega in pu), not absolute speed.
#       Steady-state value is ~0; small excursions either side are normal.
QC_BOUNDS = {
    "Vm": (0.0, 1.6),  # bus voltage magnitude (pu)
    "Va": (-4.0, 4.0),  # bus voltage angle (rad)
    "omega": (-1.0, 1.0),  # generator speed deviation (pu); steady-state ~0
    "delta": (-20.0, 20.0),  # generator angle (rad)
}

if SERIALIZE_MODE == "per_run":
    run_files = sorted(glob.glob(os.path.join(UQ_OUT_PATH, "run_*.parquet")))
    n_files = len(run_files)
    print(f"\n--- data quality ({min(QC_SAMPLE_N, n_files)} sampled runs) ---")
    print(
        f"Files:    {n_files} / {N_SAMPLES}  ({'OK' if n_files == N_SAMPLES else 'MISSING ' + str(N_SAMPLES - n_files)})"
    )

    rng_qc = np.random.default_rng(0)  # fixed seed: same 100 runs every re-run
    # rng_qc = np.random.default_rng()  # no seed: new random selection every re-run
    sample_paths = [
        run_files[i]
        for i in rng_qc.choice(n_files, size=min(QC_SAMPLE_N, n_files), replace=False)
    ]

    nan_runs, bounds_violations, col_counts = [], [], set()
    for fpath in sample_paths:
        run_id = os.path.basename(fpath).replace("run_", "").replace(".parquet", "")
        df = pd.read_parquet(fpath)
        col_counts.add(len(df.columns))

        if df.isnull().any().any():
            nan_runs.append((run_id, df.columns[df.isnull().any()].tolist()))

        for sig, (lo, hi) in QC_BOUNDS.items():
            for col in [c for c in df.columns if c.endswith(f"_{sig}") or c == sig]:
                vmin, vmax = df[col].min(), df[col].max()
                if vmin < lo or vmax > hi:
                    bounds_violations.append(
                        (run_id, col, f"{vmin:.3f}..{vmax:.3f}", f"[{lo}, {hi}]")
                    )

    if len(col_counts) == 1:
        print(f"Columns:  {col_counts.pop()} per file (consistent)")
    else:
        print(f"Columns:  INCONSISTENT: {col_counts}")

    if nan_runs:
        print(f"NaNs:     {len(nan_runs)} run(s) with NaNs")
        for run_id, cols in nan_runs[:5]:
            print(f"  run_{run_id}: {cols}")
        if len(nan_runs) > 5:
            print(f"  ... ({len(nan_runs) - 5} more)")
    else:
        print(f"NaNs:     none")

    if bounds_violations:
        print(f"Bounds:   {len(bounds_violations)} soft violation(s)")
        for run_id, col, rng_str, expected in bounds_violations[:10]:
            print(f"  run_{run_id}  {col}  {rng_str}  expected {expected}")
        if len(bounds_violations) > 10:
            print(f"  ... ({len(bounds_violations) - 10} more)")
    else:
        print(f"Bounds:   all within expected ranges")

per_run: 16000 files in /kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-v5/runs
  total on disk: 53901.88 MB  (3.37 MB/run)

--- data quality (100 sampled runs) ---
Files:    16000 / 16000  (OK)
Columns:  153 per file (consistent)
NaNs:     none
Bounds:   all within expected ranges


## sanity check plot
Loads a small subset of runs and plots one signal per (element, variable).
For full ensemble visualization use `gridkit_viv.ipynb`.

In [26]:
# === Sanity check plot: 2 runs x 1 bus signal + 1 gen signal per var ===
PLOT_MAX_RUNS = 5
PLOT_MAX_BUS_N = 2
PLOT_MAX_GEN_N = 2

from collections import defaultdict
from gridkit_utils import MONITORABLE_VARS_BY_ELEMENT

# rng_plot = np.random.default_rng(SEED)  # fixed seed: same selection every run
rng_plot = np.random.default_rng()  # no seed: new random selection every run

# Load subset depending on serialize mode
if SERIALIZE_MODE == "stacked":
    _plot_df_full = results_df
else:
    run_files = sorted(glob.glob(os.path.join(UQ_OUT_PATH, "run_*.parquet")))
    if not run_files:
        raise FileNotFoundError(f"No run_*.parquet files found in {UQ_OUT_PATH}")
    chosen_idx = sorted(
        rng_plot.choice(
            len(run_files), size=min(PLOT_MAX_RUNS, len(run_files)), replace=False
        )
    )
    frames = []
    for idx in chosen_idx:
        fpath = run_files[idx]
        run_id = int(
            os.path.basename(fpath).replace("run_", "").replace(".parquet", "")
        )
        df = pd.read_parquet(fpath)
        df.insert(0, "run_id", run_id)
        frames.append(df)
    _plot_df_full = pd.concat(frames, ignore_index=True)
    print(f"Loaded {len(chosen_idx)} run files: {[run_files[i] for i in chosen_idx]}")

param_cols = list(samples_df.columns)
skip_cols = {"run_id", "time", "Solver Status"} | set(param_cols)
mon_cols = [c for c in _plot_df_full.columns if c not in skip_cols]

all_run_ids = sorted(_plot_df_full["run_id"].unique())
plot_run_ids = list(
    rng_plot.choice(
        all_run_ids, size=min(PLOT_MAX_RUNS, len(all_run_ids)), replace=False
    )
)
print(f"Runs: {plot_run_ids}")
plot_df = _plot_df_full[_plot_df_full["run_id"].isin(plot_run_ids)]

col_groups = defaultdict(list)
for col in mon_cols:
    parts = col.split("_")
    key = ("Bus", parts[-1]) if parts[0] == "Bus" else (parts[0], parts[-1])
    col_groups[key].append(col)

for (elem, var), cols in sorted(col_groups.items()):
    max_n = PLOT_MAX_BUS_N if elem == "Bus" else PLOT_MAX_GEN_N
    plot_cols = list(rng_plot.choice(cols, size=min(max_n, len(cols)), replace=False))
    melted = plot_df[["time", "run_id"] + plot_cols].melt(
        id_vars=["time", "run_id"], var_name="signal", value_name=var
    )
    fig = px.line(
        melted,
        x="time",
        y=var,
        color="signal",
        line_group="run_id",
        title=f"{elem} {var} - {plot_cols}, {len(plot_run_ids)} runs",
        labels={"time": "Time (s)", var: var},
    )
    _ = fig.update_traces(opacity=0.7)
    _ = fig.update_layout(legend_title="signal")
    _ = fig.show()

Loaded 5 run files: ['/kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-v5/runs/run_10133.parquet', '/kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-v5/runs/run_12520.parquet', '/kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-v5/runs/run_13864.parquet', '/kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-v5/runs/run_5299.parquet', '/kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-v5/runs/run_9152.parquet']
Runs: [np.int64(13864), np.int64(10133), np.int64(12520), np.int64(9152), np.int64(5299)]


## samples sanity check
Plot sampled H values for a chosen generator, with nominal and ±1 std reference lines.

In [ ]:
# === Samples sanity check: H distribution for one generator ===
# Set to any id that appears in PARAM_SPECS, e.g. "2_1" or "23_1"
PLOT_GEN_ID = "2_1"

# --- find the spec for the chosen generator ---
_spec = next((s for s in PARAM_SPECS if s["id"] == PLOT_GEN_ID), None)
if _spec is None:
    raise ValueError(
        f"Generator id '{PLOT_GEN_ID}' not found in PARAM_SPECS. "
        f"Available: {[s['id'] for s in PARAM_SPECS]}"
    )

_col = f"Genrou_{PLOT_GEN_ID}_H"
if _col not in samples_df.columns:
    # column name may vary; fall back to first matching column
    _col = next((c for c in samples_df.columns if PLOT_GEN_ID in c), None)
    if _col is None:
        raise KeyError(
            f"No column for gen '{PLOT_GEN_ID}' in samples_df. "
            f"Columns: {list(samples_df.columns)}"
        )

_vals = samples_df[_col].values

if _spec["dist"] == "normal":
    _nominal = _spec["mean"]
    _std = _spec["std"]
    _dist_label = f"N({_nominal:.3f}, {_std:.4f})"
else:
    _nominal = _spec["nominal"]
    _std = _nominal * _spec["pct"]
    _dist_label = f"U({_nominal*(1-_spec['pct']):.4f}, {_nominal*(1+_spec['pct']):.4f})"

_plot_df = pd.DataFrame({"sample_index": range(len(_vals)), "H": _vals})

fig = px.scatter(
    _plot_df,
    x="sample_index",
    y="H",
    opacity=0.5,
    title=f"Sampled H values — Genrou {PLOT_GEN_ID}  ({_dist_label},  N={len(_vals)})",
    labels={"sample_index": "Sample index", "H": "H (inertia constant)"},
)
_ = fig.add_hline(
    y=_nominal,
    line_dash="solid",
    line_color="black",
    annotation_text=f"nominal = {_nominal:.4f}",
    annotation_position="top right",
)
_ = fig.add_hline(
    y=_nominal + _std,
    line_dash="dash",
    line_color="crimson",
    annotation_text=f"+1σ = {_nominal+_std:.4f}",
    annotation_position="top right",
)
_ = fig.add_hline(
    y=_nominal - _std,
    line_dash="dash",
    line_color="steelblue",
    annotation_text=f"-1σ = {_nominal-_std:.4f}",
    annotation_position="bottom right",
)
_ = fig.update_layout(showlegend=False)
_ = fig.show()

print(f"  mean:   {_vals.mean():.5f}  (nominal {_nominal:.4f})")
print(f"  std:    {_vals.std():.5f}  (target {_std:.4f})")
print(f"  min:    {_vals.min():.5f}")
print(f"  max:    {_vals.max():.5f}")

In [ ]:
# # === Samples histogram: all generators -- sanity check Gaussian shape ===
# from scipy.stats import norm as _scipy_norm
# import plotly.graph_objects as go

# for _s in PARAM_SPECS:
#     _gid = _s["id"]
#     _col = f"Genrou_{_gid}_H"
#     if _col not in samples_df.columns:
#         _col = next((c for c in samples_df.columns if _gid in c), None)
#     if _col is None:
#         print(f"  WARNING: no column found for gen {_gid}, skipping")
#         continue

#     _v = samples_df[_col].values
#     _mu = _s["mean"] if _s["dist"] == "normal" else _s["nominal"]
#     _sig = _s["std"] if _s["dist"] == "normal" else _s["nominal"] * _s["pct"]

#     # x range for the reference PDF
#     _x = np.linspace(_v.min(), _v.max(), 300)
#     _pdf = _scipy_norm.pdf(_x, loc=_mu, scale=_sig)

#     fig = go.Figure()
#     _ = fig.add_trace(
#         go.Histogram(
#             x=_v,
#             histnorm="probability density",
#             name="samples",
#             marker_color="steelblue",
#             opacity=0.7,
#             nbinsx=50,
#         )
#     )
#     _ = fig.add_trace(
#         go.Scatter(
#             x=_x,
#             y=_pdf,
#             mode="lines",
#             name=f"N({_mu:.3f}, {_sig:.4f})",
#             line=dict(color="crimson", width=2),
#         )
#     )
#     _ = fig.update_layout(
#         title=f"H samples — Genrou {_gid}  (N={len(_v)}, std={_sig:.4f} = {_sig/_mu*100:.0f}% of nominal)",
#         xaxis_title="H (inertia constant)",
#         yaxis_title="Probability density",
#         barmode="overlay",
#         legend_title="",
#     )
#     _ = fig.show()

# end